# Lounge scorecard: served demand + cost

**What changed from v3:** demand is site-served women (a typical lounge at the site; not catchment women, and not the lounge's own served women, which are shown as `performance_vs_site`); capture and rating left the scorecard (rating now sits inside the Huff attractiveness); cost efficiency joined; cannibalisation is a display-only what-if. Composite ≥ 0.65 is PROTECT, ≤ 0.35 SHRINK. Reference: the frozen v3 labels in `data/scenarios/baseline_v3_labels.csv`.

**The model, as agreed on 2026-10-11** (one model for lounges and growth):
1. **Demand = served women, Huff** (`src/model/served.py`). Each cell's addressable women are split, Huff-style, across every premium salon and every lounge within 20 minutes of it, across emirate borders. Weight = reviews^ALPHA × (rating/4.6)^RATING_ELASTICITY × DECAY[drive-time band], plus an outside option (women who go elsewhere or use no premium salon). Drive-time bands come from the cell's own isochrones, else from straight-line bands fitted on those. Cells our salon search never covered get an imputed stand-in competitor (emirate × density tier).
2. **Cost = site cost** (`src/features/lounges.py`): shop rent × 150 m² + an assumed fit-out written off over 5 years. Shop rent per cell from three sources (2026-10-11): DLD Ejari contracts (Dubai), ADREC 2025 district rates (Abu Dhabi), shop listings' asking rents (northern emirates), else the emirate median. Cost efficiency = site-served women (an area: net new women) per AED 1k of site cost; `cost_index` = that ÷ `COST_ANCHOR` = 14.34, one frozen UAE-wide anchor: the median lounge site at baseline (`COST_ANCHOR_WHY`; 1 = typical). It never moves when lounges open or close or the levels change.
3. **Lounge scorecard** = site demand + cost index. Site demand (`site_served`) = the women a typical lounge (median reviews and rating) would serve at this lounge's site, with this lounge closed and its siblings and rivals open. The lounge's own served women are context, not scored: `performance_vs_site` = served ÷ site-served ("The model estimates this lounge is performing 2.7x compared to its site"). Cannibalisation (the share of a lounge's women its siblings would absorb if it closed) is shown, not scored.
4. **Growth** = what-if: open a typical lounge (median lounge reviews and rating) at each area's centroid, and count the women it serves, how many it takes from our lounges, and the net new.

**Every parameter here is a first cut**, not calibrated. This notebook shows what they produce so we can decide together.

In [1]:
import sys, pathlib, statistics
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_rows", 120)
from src.baseline import run
from src.data_v3 import load_v3
from src.config import REPO_ROOT, load_baseline_assumptions
from src.model import served as S, scorecard, growth
from src.features import lounges as L
r = run(); v3 = load_v3(); a = load_baseline_assumptions(REPO_ROOT / "data/scenarios/baseline.yaml")
old = pd.read_csv(REPO_ROOT / "data/scenarios/baseline_v3_labels.csv")
old_action = dict(zip(old.id, old.action))
F = pd.DataFrame([x.model_dump() for x in r.features]).set_index("branch_id")
D = {d.branch_id: d for d in r.decisions}
AR = pd.DataFrame([x.model_dump(exclude={"cell_ids"}) for x in r.areas]).set_index("area_id")
AR["call"] = AR.index.map({d.area_id: d.action for d in r.area_decisions})

## 1. Signals and anchors

In [2]:
pd.set_option("display.max_colwidth", 200)
print(L.COST_ANCHOR_WHY, "\n")
sc = F[~F.not_scored]
print("COST_ANCHOR", L.COST_ANCHOR, "| median scored site-served women per AED 1k now:", round(sc.served_per_cost.median(), 2),
      "| median scored cost_index:", round(sc.cost_index.median(), 3))
pd.DataFrame([dict(signal=s.name, field=s.field, worst=s.worst, best=s.best, weight=s.weight, why=s.why) for s in scorecard.SIGNALS])

Cost index = women per AED 1k of annual site cost against ONE UAE-wide anchor, frozen 2026-10-11: the median site-served women per AED 1k over the 23 scored lounges at baseline (default levels, nothing closed) = 14.34. A constant, so a lounge's or area's index moves only with its own served women and site cost, never when a sibling opens, closes or grows or the levels change (the old emirate-median baseline did: SANITY_CHECKS F1). Lounges are measured on site-served women (a typical lounge at the site), areas on net new women of a what-if lounge; both against the same anchor. Re-freeze it (and say so) only when the served model or site cost changes. 

COST_ANCHOR 14.34 | median scored site-served women per AED 1k now: 14.34 | median scored cost_index: 1.0


,signal,field,worst,best,weight,why
0,demand,site_served,0.0,20000.0,1.0,"Quick first cut (2026-10-11): scores the SITE, not the lounge's popularity. A typical lounge (median reviews and rating) at the site serves about 1.4k-23k women over the scored lounges; 20k is abo..."
1,cost,cost_index,0.5,1.5,1.0,"1 = as cost-efficient as the median lounge site at baseline (COST_ANCHOR, frozen: COST_ANCHOR_WHY in src/features/lounges.py); half as good scores 0, 1.5x scores 1. Site cost = shop rent x 150 m2 ..."


## 2. v3 → new calls, with reasons

In [3]:
rows = []
for b, d in D.items():
    if d.action == "NOT SCORED": continue
    x = F.loc[b]
    reason = (f"site serves {x.site_served:,.0f} (demand {d.scores['demand']:.2f}); "
              + (f"cost index {x.cost_index:.2f} (cost {d.scores['cost']:.2f})" if x.cost_index == x.cost_index and x.cost_index is not None
                 else "no rent: cost neutral 0.5"))
    rows.append(dict(lounge=b, emirate=x.emirate, v3=old_action.get(b), new=d.action, composite=round(d.composite, 3),
                     site_served=round(x.site_served), served=round(x.served_women), perf_vs_site_context=round(x.performance_vs_site, 1),
                     confidence=d.confidence, reason=reason, weight_flips="; ".join(d.weight_flips)))
t = pd.DataFrame(rows).sort_values("composite", ascending=False).set_index("lounge")
print(t.new.value_counts().to_dict()); print(pd.crosstab(t.v3, t.new))
with pd.option_context("display.max_colwidth", 90): display(t)

{'PROTECT': 10, 'SHRINK': 9, 'HOLD': 4}
new      HOLD  PROTECT  SHRINK
v3                            
HOLD        3        7       5
PROTECT     0        3       0
SHRINK      1        0       4


,emirate,v3,new,composite,site_served,served,perf_vs_site_context,confidence,reason,weight_flips
lounge,,,,,,,,,,
al-dhafra,Abu Dhabi,HOLD,PROTECT,1.000,22852,17634,0.8,high,"site serves 22,852 (demand 1.00); cost index 5.48 (cost 1.00)",
baniyas,Abu Dhabi,HOLD,PROTECT,0.826,13036,11086,0.9,high,"site serves 13,036 (demand 0.65); cost index 2.33 (cost 1.00)",
al-ain,Abu Dhabi,PROTECT,PROTECT,0.754,10173,18652,1.8,high,"site serves 10,173 (demand 0.51); cost index 2.68 (cost 1.00)",
shahama,Abu Dhabi,HOLD,PROTECT,0.747,9865,9865,1.0,medium,"site serves 9,865 (demand 0.49); cost index 1.95 (cost 1.00)",
al-taif-mall,Fujairah,PROTECT,PROTECT,0.725,8980,6828,0.8,medium,"site serves 8,980 (demand 0.45); cost index 3.86 (cost 1.00)",
shakhbout-city,Abu Dhabi,HOLD,PROTECT,0.717,8685,12743,1.5,medium,"site serves 8,685 (demand 0.43); cost index 1.67 (cost 1.00)",
ras-al-khaimah,Ras Al Khaimah,PROTECT,PROTECT,0.699,7946,7328,0.9,low,"site serves 7,946 (demand 0.40); cost index 2.08 (cost 1.00)",
al-jada,Sharjah,HOLD,PROTECT,0.698,7912,5853,0.7,low,"site serves 7,912 (demand 0.40); cost index 1.55 (cost 1.00)",
al-falah,Abu Dhabi,HOLD,PROTECT,0.697,7866,9510,1.2,low,"site serves 7,866 (demand 0.39); cost index 2.07 (cost 1.00)",


In [4]:
ch = t[t.v3 != t.new]
print(len(ch), "of", len(t), "lounges change call:"); ch[["v3", "new", "composite", "reason"]]

13 of 23 lounges change call:


,v3,new,composite,reason
lounge,,,,
al-dhafra,HOLD,PROTECT,1.000,"site serves 22,852 (demand 1.00); cost index 5.48 (cost 1.00)"
baniyas,HOLD,PROTECT,0.826,"site serves 13,036 (demand 0.65); cost index 2.33 (cost 1.00)"
shahama,HOLD,PROTECT,0.747,"site serves 9,865 (demand 0.49); cost index 1.95 (cost 1.00)"
shakhbout-city,HOLD,PROTECT,0.717,"site serves 8,685 (demand 0.43); cost index 1.67 (cost 1.00)"
al-jada,HOLD,PROTECT,0.698,"site serves 7,912 (demand 0.40); cost index 1.55 (cost 1.00)"
al-falah,HOLD,PROTECT,0.697,"site serves 7,866 (demand 0.39); cost index 2.07 (cost 1.00)"
zawaya-walk,HOLD,PROTECT,0.689,"site serves 7,569 (demand 0.38); cost index 1.77 (cost 1.00)"
mohammed-bin-zayed-city,SHRINK,HOLD,0.445,"site serves 4,581 (demand 0.23); cost index 1.16 (cost 0.66)"
ministries-complex,HOLD,SHRINK,0.235,"site serves 3,919 (demand 0.20); cost index 0.77 (cost 0.27)"


## 3. Cannibalisation what-if (display only)

Close each lounge, re-split: `absorbed` = the share of its served women its siblings would take over. Not scored.

In [5]:
cn = F[~F.not_scored][["emirate", "served_women", "absorbed_share"]].assign(absorbed_women=lambda x: x.served_women * x.absorbed_share)
cn["call"] = cn.index.map(lambda b: D[b].action)
cn.sort_values("absorbed_share", ascending=False).round(2)

,emirate,served_women,absorbed_share,absorbed_women,call
branch_id,,,,,
noya-plaza,Abu Dhabi,1385.72,0.30,419.15,SHRINK
westyas,Abu Dhabi,3160.09,0.27,850.06,SHRINK
khalifa-city-a,Abu Dhabi,16679.26,0.20,3308.14,HOLD
mohammed-bin-zayed-city,Abu Dhabi,4056.92,0.19,784.20,HOLD
shahama,Abu Dhabi,9865.31,0.19,1879.86,PROTECT
shakhbout-city,Abu Dhabi,12743.27,0.19,2381.51,PROTECT
al-maqta,Abu Dhabi,6989.73,0.18,1236.37,HOLD
ministries-complex,Abu Dhabi,8397.89,0.16,1370.32,SHRINK
al-falah,Abu Dhabi,9509.65,0.15,1447.75,PROTECT


### Performance vs site (context, not scored)

`performance_vs_site` = the lounge's own served women ÷ what a typical lounge would serve at its site. Above 1, the lounge out-performs its site (mostly its review count). Shown, not scored.

In [6]:
pv = F[~F.not_scored][["emirate", "review_count", "served_women", "site_served", "performance_vs_site"]].assign(call=lambda x: x.index.map(lambda b: D[b].action))
print("Spearman performance_vs_site vs reviews:", round(pv.performance_vs_site.corr(pv.review_count, method="spearman"), 2))
pv.sort_values("performance_vs_site", ascending=False).round(2)

Spearman performance_vs_site vs reviews: 0.95


,emirate,review_count,served_women,site_served,performance_vs_site,call
branch_id,,,,,,
khalifa-city-a,Abu Dhabi,2465,16679.26,6144.21,2.71,HOLD
khaleej-al-arabi,Abu Dhabi,1753,3083.19,1358.59,2.27,SHRINK
ministries-complex,Abu Dhabi,1217,8397.89,3919.20,2.14,SHRINK
al-ain,Abu Dhabi,1332,18651.84,10173.46,1.83,PROTECT
al-maqta,Abu Dhabi,1295,6989.73,4159.11,1.68,HOLD
shakhbout-city,Abu Dhabi,1277,12743.27,8685.03,1.47,PROTECT
al-falah,Abu Dhabi,951,9509.65,7865.50,1.21,PROTECT
city-walk,Dubai,857,2155.24,1905.03,1.13,SHRINK
shahama,Abu Dhabi,757,9865.31,9865.31,1.00,PROTECT


## 4. Composites against the lines

In [7]:
print(t.composite.describe().round(3))
print("within 0.05 of a line:", list(t.index[(abs(t.composite - scorecard.PROTECT_AT) < 0.05) | (abs(t.composite - scorecard.SHRINK_AT) < 0.05)]))
print("no cost index (cost scores neutral 0.5):", list(F.index[~F.not_scored & F.cost_index.isna()]))
print("confidence:", t.confidence.value_counts().to_dict())
dem = next(s for s in scorecard.SIGNALS if s.name == "demand")
print("with cost neutral (0.5), PROTECT needs site_served >=", round((2 * scorecard.PROTECT_AT - 0.5) * dem.best),
      "and SHRINK needs site_served <=", round((2 * scorecard.SHRINK_AT - 0.5) * dem.best))

count    23.000
mean      0.440
std       0.311
min       0.034
25%       0.118
50%       0.404
75%       0.708
max       1.000
Name: composite, dtype: float64
within 0.05 of a line: ['ras-al-khaimah', 'al-jada', 'al-falah', 'zawaya-walk', 'mirdif-35', 'al-maqta']
no cost index (cost scores neutral 0.5): []
confidence: {'high': 11, 'medium': 6, 'low': 6}
with cost neutral (0.5), PROTECT needs site_served >= 16000 and SHRINK needs site_served <= 4000


## Open questions for review

1. **All 23 scored lounges now have a cost index.** One UAE-wide anchor (`COST_ANCHOR` = 14.34, the median lounge site, so the median index is 1.00) gives al-taif-mall one too (3.86); no lounge scores a neutral 0.5. Were a rent to go missing, PROTECT would need a site serving 16,000+ women and SHRINK 4,000 or fewer.
2. **13 of 23 lounges change call from v3.** al-dhafra, baniyas, shahama, shakhbout-city, al-jada, al-falah and zawaya-walk go HOLD → PROTECT. mohammed-bin-zayed-city goes SHRINK → HOLD. ministries-complex, nad-al-sheba, jumeirah-park, al-barsha and city-walk go HOLD → SHRINK. New counts: PROTECT 10, HOLD 4, SHRINK 9.
3. **Cost now mostly sorts Dubai from the rest.** Against one anchor, every PROTECT has a cost index of 1.55 or more (cost score 1.00) and every SHRINK 0.85 or less; only 4 lounges score cost between 0 and 1. All 5 Dubai lounges sit below 1 (city-walk 0.17 to mirdif-35 0.98): 4 SHRINK and mirdif-35 HOLD. Is the 0.5–1.5 scale too narrow now the anchor is UAE-wide?
4. **Scoring the site, not the lounge, pulls strong performers down** (the performance-vs-site table after section 3). khalifa-city-a is HOLD although the model estimates this lounge is performing 2.7x compared to its site; ministries-complex (2.1x) and khaleej-al-arabi (2.3x) are SHRINK. At the other end noya-plaza performs 0.3x its site. This is context, not scored: performance vs site tracks review count (Spearman 0.95), which is why it was left out.
5. **Many calls sit near a line.** 6 lounges are within 0.05 of a line (ras-al-khaimah, al-jada, al-falah, zawaya-walk, mirdif-35, al-maqta), and 6 are low confidence. Four of the 10 PROTECTs clear 0.65 by under 0.05 (ras-al-khaimah 0.699 down to zawaya-walk 0.689, which flips to HOLD with the cost weight ×0.75); al-maqta is HOLD at 0.350, on the SHRINK line.
6. **Cannibalisation is display only.** It is highest at noya-plaza (0.30) and westyas (0.27), 0.01 at every Dubai lounge, and 0.00 at al-ain, al-dhafra, ras-al-khaimah and al-taif-mall. Is showing it, without scoring it, enough?